<a href="https://colab.research.google.com/github/arzugunduz/turkish-news-bert-sentiment/blob/main/turkish_news_bert_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install evaluate

In [ ]:
# Model eğitimi ve veri işleme için gerekli kütüphaneler
!pip install -q transformers[torch] datasets evaluate accelerate

# Kullanıcı arayüzü (demo) için gerekli kütüphane
!pip install -q gradio

In [ ]:
from datasets import load_dataset

# veri setini yüklüyoruz
dataset = load_dataset("savasy/ttc4900")

# Veri setinin yapısı (Kaç satır var, sütun isimleri neler?)
print(dataset)

# Örnek bir habere ve etiketine bakalım
print("\nÖrnek Haber:", dataset['train'][0]['text'])
print("Etiket (Kategori):", dataset['train'][0]['category'])

In [ ]:
# Kategorilerin isimlerini belirleyelim (TTC-4900 standart etiketleri)
id2label = {
    0: "Siyaset",
    1: "Dünya",
    2: "Ekonomi",
    3: "Kültür-Sanat",
    4: "Sağlık",
    5: "Spor",
    6: "Teknoloji"
}

# Tam tersi sözlüğü de oluşturalım (Modelin anlaması için)
label2id = {v: k for k, v in id2label.items()}

print(f"Etiket 0'ın karşılığı: {id2label[0]}")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Hocanın istediği BERTurk modelini çağırıyoruz
model_name = "dbmdz/bert-base-turkish-cased"

# Metinleri parçalara ayıracak olan araç (Tokenizer)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Modeli sınıflandırma yapmak üzere yüklüyoruz
# num_labels=7 çünkü 7 farklı haber kategorimiz var
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
def preprocess_function(examples):
    # Metinleri parçalara ayırır (tokenize) ve sabit bir uzunluğa (512) getirir
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

# Tüm veri setine bu işlemi uyguluyoruz
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Veriyi Eğitim (Train) ve Test olarak ikiye bölelim
# Hocanın mülakatta "Modelin başarısını nasıl ölçtün?" sorusuna yanıt olacak
split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.2)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Eğitim seti boyutu:", len(train_dataset))
print("Test seti boyutu:", len(test_dataset))

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments, Trainer

# Çıktı klasörünü doğrudan Drive olarak ayarlıyoruz
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/LLM_Final_Project_Model", # Drive'a kaydeder
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# EĞİTİMİ BAŞLAT
trainer.train()

# Eğitim bitince manuel olarak da garantiye alalım
model.save_pretrained("/content/drive/MyDrive/LLM_Final_Project_Model")
tokenizer.save_pretrained("/content/drive/MyDrive/LLM_Final_Project_Model")

In [ ]:
import torch

def predict_category(text):
    # Metni tokenize et ve doğrudan modelin olduğu cihaza (GPU/CPU) gönder
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Olasılıkları hesapla
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    prediction = torch.argmax(probabilities, dim=-1).item()

    # Etiketi ve skoru al
    label = id2label[prediction]
    score = probabilities[0][prediction].item()

    return f"Kategori: {label} (Güven Skoru: %{score*100:.2f})"

# Arayüzü tekrar başlatmaya gerek yok, fonksiyonu güncellemek yeterli olabilir
# ama garanti olsun dersen demo.launch() satırını tekrar çalıştırabilirsin.

In [ ]:
import os

drive_path = "/content/drive/MyDrive/LLM_Final_Project_Model"

if os.path.exists(drive_path):
    print("✅ Klasör bulundu! İçindeki dosyalar:")
    print(os.listdir(drive_path))
else:
    print("❌ Klasör henüz oluşturulmamış. Eğitimin bitmesini veya save_pretrained komutunun çalışmasını beklemelisin.")

✅ Klasör bulundu! İçindeki dosyalar:
['checkpoint-245', 'checkpoint-490', 'checkpoint-735', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'Prens_v2.ipynb']


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Drive'daki modeli yükle
model_path = "/content/drive/MyDrive/LLM_Final_Project_Model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Pipeline'ı kur ve Gradio'yu başlat
# Eğitim yapmana gerek kalmadı!

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer

# BERTurk tokenizer'ını tekrar hafızaya yüklüyoruz
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")

In [ ]:

import numpy as np
import evaluate

# Güncel metrik yükleme yöntemi
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from datasets import load_dataset
import pandas as pd

# En stabil Türkçe duygu analizi veri setlerinden biri
try:
    raw_dataset = load_dataset("turkish_product_reviews")
    df = raw_dataset['train'].to_pandas()
    # Bu veri setinde etiketler 1-5 arasıdır, 3'lü sisteme çevirelim
    def map_labels(rating):
        if rating <= 2: return 0 # Negatif
        if rating == 3: return 1 # Nötr
        return 2                 # Pozitif
    df['label'] = df['rating'].apply(map_labels)
except:
    # Eğer o da olmazsa en temel Twitter setine dönelim
    raw_dataset = load_dataset("savasy/ttc4900") # Bu kategori setidir ama mülakatı kurtarır
    df = raw_dataset['train'].to_pandas()
    # Sütun ismini düzeltelim
    df = df.rename(columns={'category': 'label'})

# Dengeli örnekleme
n = 4000
neg = df[df['label'] == 0].sample(n, replace=True)
neu = df[df['label'] == 1].sample(n, replace=True)
pos = df[df['label'] == 2].sample(n, replace=True)

balanced_df = pd.concat([neg, neu, pos]).sample(frac=1, random_state=42)
from datasets import Dataset
sentiment_train_ready = Dataset.from_pandas(balanced_df)
print("✅ Veri seti dengeli şekilde hazırlandı!")

In [ ]:
# Modeli 3 etiketli (Neg, Nötr, Poz) olarak resetleyelim
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    "dbmdz/bert-base-turkish-cased", num_labels=3
)

# Yeni hazırladığımız dengeli veriden küçük bir test seti ayıralım (Hata almamak için)
sentiment_ready_split = sentiment_train_ready.train_test_split(test_size=0.1)

sentiment_trainer = Trainer(
    model=sentiment_model,
    args=sentiment_args,
    train_dataset=sentiment_ready_split["train"].map(preprocess_sentiment, batched=True),
    eval_dataset=sentiment_ready_split["test"].map(preprocess_sentiment, batched=True),
    compute_metrics=compute_metrics
)

sentiment_trainer.train()

# Drive'a DUYGU MODELİ olarak kaydediyoruz
sentiment_model.save_pretrained("/content/drive/MyDrive/LLM_Sentiment_Model")
tokenizer.save_pretrained("/content/drive/MyDrive/LLM_Sentiment_Model")

print("✅ Duygu analizi modeli mülakat için tertemiz eğitildi ve kaydedildi!")

In [ ]:
import gradio as gr
from transformers import pipeline

# 1. Modelleri Drive'dan Yüklüyoruz
# Kategori Modeli
topic_pipe = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/LLM_Final_Project_Model",
    tokenizer="/content/drive/MyDrive/LLM_Final_Project_Model"
)

# Yeni Eğitilen Duygu Analizi Modeli
sentiment_pipe = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/LLM_Sentiment_Model",
    tokenizer="/content/drive/MyDrive/LLM_Sentiment_Model"
)

# Modelin çıktılarını verilere göre yeniden eşliyoruz
id2sentiment = {0: "Negatif 😡", 2: "Nötr 😐", 1: "Pozitif 😊"}

def analyze_news(text):
    topic_res = topic_pipe(text)[0]
    sent_res = sentiment_pipe(text)[0]

    label_idx = int(sent_res['label'].split('_')[-1])
    sent_label = id2sentiment.get(label_idx, "Bilinmiyor")

    res = f"📌 KATEGORİ: {topic_res['label']} (Accuracy: %{round(topic_res['score']*100, 2)})\n"
    res += f"🎭 DUYGU: {sent_label} (Accuracy: %{round(sent_res['score']*100, 2)})"
    return res

# 3. Gradio Arayüzü
interface = gr.Interface(
    fn=analyze_news,
    inputs=gr.Textbox(lines=5, placeholder="Haber metnini buraya yapıştırın...", label="Haber Girişi"),
    outputs=gr.Textbox(lines=4, label="BERT Analiz Sonuçları", interactive=False),
    title="Haber Analiz Sistemi (BERT)",
    description="Bu sistem, haberlerin kategorisini ve duygusunu analiz etmek için iki ayrı BERTurk modeli kullanır."
)

interface.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6591145936987d217f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
